# Talk to your agent from your phone, on Google Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/courses/sovereign-agent-book/book/textbook/appendices/profrod_sovereign_agent_textbook_ch09_phone_on_colab_v1.ipynb)

> **Learn with Prof Rod**: *Build Your Always-On AI Agent From Scratch*. Read the full book at [profrod.ai/book](https://profrod.ai/book) and join the [learner community](https://profrod.ai/community).

This notebook runs Chapter 9's phone channel, the code you built, on Google Colab. Lucy sends a message to her own Telegram bot; the agent in this notebook reads it, runs the opening procedure and replies.

1. **Setup** fetches the course and checks the Python version.
2. **The offline check** runs Chapter 9's checkpoint with no credentials and no network use by the agent.
3. **Your bot** needs two Colab secrets: the bot credential and your numeric Telegram user ID.
4. **Serve** answers your messages for a bounded number of rounds.

Everything the agent keeps lives in `/content/lucy-phone`, which disappears when the Colab runtime is reset. The last section shows how to save it to Google Drive and bring it back.

## 1. Setup

On Colab this clones the public course repository into `/content` and installs the one package the chapters use. Elsewhere, start this notebook from the course folder.

In [ ]:
import json
import os
import runpy
import subprocess
import sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    checkout = Path("/content/profrodai-resources")
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/profrodai/profrodai-resources",
             str(checkout)],
            check=True,
        )
    os.chdir(checkout / "courses" / "sovereign-agent-book")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic==2.13.4"], check=True)
assert Path("book/textbook/learner").is_dir(), "start this notebook from the course folder"
assert sys.version_info[:2] >= (3, 12), "the chapters need Python 3.12 or newer"
print("Python", sys.version.split()[0], "in", Path.cwd())

## 2. The offline check

Chapter 9's checkpoint with its local bot: intake, a replayed batch, a conflicting claim, two answered requests, a lost reply that stays unknown, and a stopped cell. No credential is used.

In [ ]:
checkpoint = "book/textbook/checkpoints/profrod_sovereign_agent_ch09_telegram_messaging_checkpoint.py"
result = subprocess.run([sys.executable, checkpoint], capture_output=True, text=True)
print(result.stdout[-1500:] or result.stderr[-1500:])
assert result.returncode == 0, "the offline check failed; read its output above"

## 3. Your bot and your ID

Use a **dedicated** teaching bot and a private chat you start yourself.

1. In Telegram, ask **@BotFather** for `/newbot` and copy the bot credential it gives you.
2. In Colab, open the **Secrets** panel (the key icon on the left), add a secret named `SOVEREIGN_AGENT_TELEGRAM_TOKEN` with that credential, and switch on **Notebook access**.
3. Run the next cell. It shows a one-time challenge; send exactly that text to your bot from your own account. It prints your numeric user ID and changes nothing.
4. Add a second secret, `SOVEREIGN_AGENT_OPERATORS`, with that number, and give it notebook access too.

Secrets are read inside this notebook and never printed. Do not paste the credential into a cell.

In [ ]:
CHANNEL = runpy.run_path("book/textbook/learner/profrod_sovereign_agent_ch09_messaging_learner.py")
secret = CHANNEL["secret"]
if secret("SOVEREIGN_AGENT_TELEGRAM_TOKEN"):
    identity = runpy.run_path(
        "book/textbook/appendices/profrod_sovereign_agent_textbook_telegram_identity_v1.py"
    )
    identity["main"]()
else:
    print("No SOVEREIGN_AGENT_TELEGRAM_TOKEN secret yet; add it as described above, then rerun.")

## 4. Serve

Each round waits up to twenty seconds for a message, answers it and delivers the reply, so fifteen rounds keep the channel open for about five minutes. Send your bot: *Prepare replenishment drafts from current stock. State USD amounts.*

`MODEL = "offline"` uses the chapter's fixture model, which tests the channel without a language model. For real answers, set `MODEL` to a Claude model such as `"claude-haiku-4-5-20251001"` and add an `ANTHROPIC_API_KEY` secret; the run stops spending at one US dollar.

Stop the cell whenever you like. A turn in progress is finished with a reply asking Lucy to send it again, and the next run picks up anything left waiting.

In [ ]:
MODEL = "offline"
ROUNDS = 15

token = secret("SOVEREIGN_AGENT_TELEGRAM_TOKEN")
operators = frozenset(
    int(actor) for actor in secret("SOVEREIGN_AGENT_OPERATORS").split(",") if actor.strip().isdigit()
)
if not (token and operators):
    print("Add both secrets and give this notebook access to them, then rerun this cell.")
else:
    LOOP, SKILLS = CHANNEL["LOOP"], CHANNEL["SKILLS"]
    if MODEL == "offline":
        model_factory = SKILLS["OfflineShopModel"]
    else:
        claude = runpy.run_path("book/textbook/experiments/profrod_sovereign_agent_claude_messages_v1.py")
        client = claude["Claude"](max_usd=1.0)

        def model_factory():
            return claude["LoopModel"](client, MODEL, LOOP["ModelTurn"], LOOP["ToolCall"])

    ROOT = Path("/content/lucy-phone") if ON_COLAB else Path("lucy-phone")
    ROOT.mkdir(exist_ok=True)
    db, queue = CHANNEL["open_channel"](ROOT / "agent.sqlite")
    CHANNEL["activate_opening_skill"](db)
    bot = CHANNEL["Telegram"](token)
    try:
        summary = CHANNEL["serve"](db, queue, bot, operators, model_factory, rounds=ROUNDS)
        print(json.dumps(summary, indent=2))
        if MODEL != "offline":
            print("Claude spend:", client.report()["costUsd"], "USD")
    finally:
        db.close()

## 5. Keep the agent's memory between sessions

`/content` is erased when the runtime resets. SQLite's own backup copies the whole database consistently, even while it is open, so saving to Google Drive is one call. Restore it into `/content` before serving again; the agent should not run its database directly on Drive.

In [ ]:
import sqlite3

def copy_database(source, target):
    with sqlite3.connect(source) as origin, sqlite3.connect(target) as copy:
        origin.backup(copy)

SAVE_TO_DRIVE = False
RESTORE_FROM_DRIVE = False
if ON_COLAB and (SAVE_TO_DRIVE or RESTORE_FROM_DRIVE):
    from google.colab import drive

    drive.mount("/content/drive")
    saved = Path("/content/drive/MyDrive/lucy-phone-agent.sqlite")
    live = Path("/content/lucy-phone/agent.sqlite")
    if SAVE_TO_DRIVE:
        copy_database(live, saved)
        print("Saved to", saved)
    if RESTORE_FROM_DRIVE:
        live.parent.mkdir(exist_ok=True)
        copy_database(saved, live)
        print("Restored from", saved)